# Unified Colab launcher — LLM clustering heuristics

This is the main notebook for running the project from Google Colab.

The repo contains the backend code, configs, prompt builders, objective evaluators, and pipeline script. This notebook is only the launcher/control panel.

Recommended workflow:

1. Select the run and key parameters in the control panel.
2. Mount Drive.
3. Clone the GitHub repo.
4. Install and optionally run tests.
5. Build a runtime config from the YAML defaults plus notebook overrides.
6. Run a 1-attempt smoke test first.
7. If the smoke test works, set `SMOKE_TEST = False` and launch the full run.
8. Auto-download the final artifact zip.


## 1. Run control panel

Most changes should happen here. Each variable has an inline comment explaining what it controls.


In [ ]:
# ============================================================
# RUN CONTROL PANEL
# ============================================================

# -------------------------
# GitHub backend repo
# -------------------------
ORG = "TM-HESSO-202526"
REPO = "llm-clustering-heuristics"
BRANCH = "main"

# -------------------------
# Main run choice
# -------------------------
RUN = "A"                       # "A" = SSE/free centers, "B" = p-median/data-point centers, "C" = radius-volume/data-point medoid centers
SMOKE_TEST = False              # True = force exactly 1 LLM attempt in config
MAX_LLM_CALLS = 20              # Used only when SMOKE_TEST = False and FAMILY_FOCUS_MODE = False

# -------------------------
# LLM provider and sampling
# -------------------------
LLM_PROVIDER = "groq"
LLM_MODEL = "llama-3.3-70b-versatile"
TEMPERATURE = 0.8
TOP_P = 1.0

# -------------------------
# Groq key/rate-limit behavior
# -------------------------
GROQ_MAX_KEYS = 10
LLM_CALLS_PER_MINUTE_PER_KEY = 2
LLM_REQUEST_TIMEOUT_S = 60
MAX_429_RETRIES = 100
MAX_REQUEST_ERROR_RETRIES = 5

# -------------------------
# Evolution/search behavior
# -------------------------
SELECTION_STRATEGY = "1+1"       # "1+1" = elitist best-so-far parent; "1,1" = latest sequential parent
HISTORY_LIMIT = 20               # Number of previous attempts summarized inside the prompt history

# -------------------------
# Invalid-parent redesign behavior, matched to the clustering launcher
# -------------------------
INVALID_PARENT_REDESIGN = True   # If no full-valid parent exists, use a special redesign prompt for invalid/partial parents
REDESIGN_ON_ANY_INVALID_BEFORE_FULL_VALID = True  # Trigger redesign on invalid/partial parents before first full-valid heuristic
REDESIGN_ON_TIMEOUT_PARENT = True # Trigger redesign when the selected parent has timeout/runtime failures
HIDE_INVALID_PARENT_CODE = False # False = expose invalid/partial parent code for diagnosis; True = hide it from the LLM

# -------------------------
# Historical family avoidance
# When True, injects the fixed historical avoidance block into the LLM prompt.
# -------------------------
HISTORICAL_FAMILY_AVOIDANCE = False

# -------------------------
# Family-focus / island exploitation mode
# -------------------------
# When True, the run is split into one local 1+1/1,1 block per family.
# Each family block has its own local parent and local history.
# At the end, the backend compares the best candidate from each family.
#
# Total LLM calls in this mode = FAMILY_FOCUS_CALLS_PER_FAMILY × number of enabled families for the selected RUN.
# Recommended use: set HISTORICAL_FAMILY_AVOIDANCE = False, because historical avoidance is the discovery stage and family focus is the exploitation stage.
FAMILY_FOCUS_MODE = True
FAMILY_FOCUS_CALLS_PER_FAMILY = 20

FAMILY_FOCUS_PLANS = {
    "A": [
        {
            "id": "recursive_geometric_partition",
            "enabled": True,
            "name": "Recursive / hierarchical geometric partitioning",
            "objective": "Split the dataset into spatial regions, allocate center budget through the partition tree, construct initial free centers from the regions, then apply only bounded SSE-compatible refinement.",
            "description": "Historical-avoidance discovery runs repeatedly produced recursive or hierarchical partitioning. This family gave the best Run A signal when the partitioning stayed scalable.",
            "strict_constraints": [
                "The recursive or hierarchical partition must be the main construction mechanism.",
                "Do not simply return random, k-means++, or farthest-first centers followed by refinement.",
                "Avoid deep recursion that creates empty regions or wastes centers.",
                "The algorithm must explicitly allocate the p centers across regions.",
                "Each region should contribute centers based on its size, spread, variance, or geometric extent.",
                "Lloyd-style refinement is allowed, but do not rely on it as the sole source of quality.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
        {
            "id": "density_grid_local_density",
            "enabled": True,
            "name": "Density-grid / local-density construction",
            "objective": "Use grid cells, bins, or scalable local-density summaries to place free centers in dense and spatially important regions while preserving coverage.",
            "description": "Density/local-density families appeared repeatedly in Run A discovery. They were weaker than recursive partitioning but promising enough to test with focused exploitation.",
            "strict_constraints": [
                "The density or grid structure must determine where centers are initially placed.",
                "Do not compute a full dense n by n distance matrix.",
                "Do not use O(n^2) density estimation on large instances.",
                "Use scalable bins, sampled density estimates, cell summaries, or local neighborhood approximations.",
                "Do not collapse into plain random center sampling, k-means++ seeding, or farthest-first seeding.",
                "Lloyd-style refinement is allowed, but do not rely on it as the sole source of quality.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
        {
            "id": "spatially_dispersed_control",
            "enabled": True,
            "name": "Spatially dispersed / spread construction control",
            "objective": "Construct free centers by explicitly balancing spatial spread and coverage, without relying on plain farthest-first or k-means++ as the whole mechanism.",
            "description": "Spread/dispersed methods produced valid candidates and are useful as a control family, but they are close to known farthest-first behavior.",
            "strict_constraints": [
                "The main mechanism must be more than plain farthest-first seeding.",
                "Do not simply choose farthest points and then rely only on refinement.",
                "Use explicit coverage balance, regional quotas, dispersion pressure, or staged core/fringe logic.",
                "Avoid random restarts as the main source of diversity.",
                "Lloyd-style refinement is allowed, but do not rely on it as the sole source of quality.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
    ],
    "B": [
        {
            "id": "geometric_voronoi_medoid_refinement",
            "enabled": True,
            "name": "Geometric / Voronoi medoid refinement",
            "objective": "Select data-point medoids, build Voronoi cells around them, and refine each cell by replacing medoids with better representative data points.",
            "description": "Run B discovery produced a very strong candidate in this family. The useful mechanism was selected-point medoids plus Voronoi-cell representative refinement.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "The Voronoi/assignment cells must be used to guide medoid replacement.",
                "Medoid refinement must select actual points from X, not computed averages.",
                "Do not reduce the method to generic random medoid swaps or exhaustive PAM-style swap search.",
                "Do not rely solely on Lloyd-style centroid updates or centroid drift; final centers must remain selected data points.",
                "The method should improve selected medoids by cell-level contribution, representative quality, or assignment cost.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
        {
            "id": "density_neighborhood_medoid_construction",
            "enabled": True,
            "name": "Density-neighborhood medoid construction",
            "objective": "Construct medoids from local density, neighborhood coverage, or demand concentration while keeping centers as selected data points.",
            "description": "Density/neighborhood methods appeared often in Run B. They were weaker than Voronoi medoid refinement but are a plausible medoid-specific alternative family.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "Do not compute dense all-pairs neighborhoods for large instances.",
                "Use scalable density estimates, sampled neighborhoods, grid summaries, or local representative candidates.",
                "Do not rely on random medoid replacement as the main mechanism.",
                "Do not rely solely on Lloyd-style centroid updates or centroid drift; final centers must remain selected data points.",
                "The selected medoids should reflect local demand, density, coverage, or representative pressure.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
        {
            "id": "spread_farthest_medoid_control",
            "enabled": True,
            "name": "Spread / farthest-medoid construction control",
            "objective": "Select data-point medoids using spatial spread and coverage balance, then apply only bounded selected-point repair.",
            "description": "Farthest-medoid and spread-like methods are close to known baselines, but they are useful as a control family for Run B.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "Do not use plain farthest-first medoid selection alone.",
                "The method must add coverage-aware, contribution-aware, or region-aware logic beyond simple max-min distance.",
                "Do not rely on exhaustive PAM-style swap search.",
                "Do not rely solely on Lloyd-style centroid updates or centroid drift; final centers must remain selected data points.",
                "Any refinement must preserve selected-point medoids.",
                "The method must remain robust in higher-dimensional settings, especially d=4.",
            ],
        },
    ],
    "C": [
        {
            "id": "distance_medoid_radius_minimax",
            "enabled": True,
            "name": "Distance/farthest medoid selection with radius-aware minimax refinement",
            "objective": "Select data-point medoids using distance coverage, then refine medoids by directly reducing the maximum radius or radius^d contribution inside bad clusters.",
            "description": "The best Run C 1,1 candidate followed this pattern: selected-point medoids, nearest assignment, identify high-radius clusters, and replace medoids to reduce worst radii.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "Do not optimize SSE or average squared distance as the main objective.",
                "The refinement criterion must use Euclidean radius, maximum cluster distance, or radius^d contribution.",
                "Do not rely only on mean compactness or nearest-center assignment quality.",
                "Identify clusters with large radius or radius^d contribution and repair those first.",
                "Avoid returning empty centers; all p centers should remain active.",
                "The method must be robust in d=4, where radius^d strongly penalizes large radii.",
            ],
        },
        {
            "id": "high_radius_cluster_splitting_repair",
            "enabled": True,
            "name": "High-radius cluster splitting / medoid repair",
            "objective": "Detect clusters with the largest radius-volume contribution, split or repair them using selected data-point medoids, and reallocate centers without wasting center budget.",
            "description": "Several Run C candidates tried to repair bad high-radius clusters. This family focuses that idea explicitly and tests whether the LLM can make it stable and scalable.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "The main mechanism must focus on the worst radius or radius^d clusters.",
                "Do not merely perform generic medoid replacement everywhere.",
                "When splitting a bad cluster, ensure the total number of centers remains exactly p.",
                "Avoid empty clusters and duplicated medoids.",
                "Use radius-aware criteria to choose replacement medoids.",
                "The method must control large radii in d=4.",
            ],
        },
        {
            "id": "nucleated_radius_volume_reduction",
            "enabled": True,
            "name": "Nucleated radius-volume reduction",
            "objective": "Grow or place medoid nuclei in high-radius/high-contribution regions, then expand and repair assignments to reduce radius-volume cost.",
            "description": "Nucleation appeared repeatedly in Run C discovery. It was not the best family, but it is a distinct radius-specific direction worth testing against the medoid-repair families.",
            "strict_constraints": [
                "Final centers must be selected input data points.",
                "Nuclei must be chosen because they reduce coverage radius or radius-volume contribution, not because of random spread alone.",
                "Do not use free-center movement.",
                "Do not optimize SSE or average distance as the main criterion.",
                "After nucleation, repair high-radius assigned clusters using selected-point medoid replacements.",
                "Avoid empty centers and duplicated centers.",
                "The method must be robust in d=4, where radius^d strongly penalizes large radii.",
            ],
        },
        {
    "id": "radius_covering_medoid_pivot_control",
    "enabled": False,
    "name": "Radius-covering medoid pivoting control",
    "objective": "Use direct radius-covering medoid selection and bounded pivot/replacement rules as a control family for the Run C objective.",
    "description": "Many Run C generated heuristics naturally fell into radius-covering behavior. This block can be used as a control, but it is not the main novelty family.",
    "strict_constraints": [
        "Final centers must be selected input data points.",
        "Use radius or radius^d contribution directly, not SSE.",
        "Do not rename a generic nearest-center assignment loop as radius-aware.",
        "Medoid pivots must be selected to reduce worst-cluster radius or radius-volume contribution.",
        "Avoid empty centers and duplicated centers.",
    ],
},
    ],
}

# -------------------------
# Evaluation/runtime behavior
# -------------------------
GLOBAL_SEED = 12345
CANDIDATE_TIMEOUT_S = 180.0

# -------------------------
# Sampling/decomposition mode
# -------------------------
SAMPLING_MODE = False
SAMPLING_MAX_XP = 10

# -------------------------
# Search/probe/final instance protocol
# -------------------------
SEARCH_SPECS = [                 # Instances used inside the LLM search loop; changing this changes what the LLM optimizes against
    {"instance_id": 1, "d": 2, "p": 20},
    {"instance_id": 1, "d": 2, "p": 40},
    {"instance_id": 1, "d": 2, "p": 70},
]

PROBE_SPECS = [                  # Extra generalization checks used after a candidate is valid on search instances
    {"instance_id": 1, "d": 2, "p": 100},
    {"instance_id": 1, "d": 3, "p": 40},
    {"instance_id": 1, "d": 4, "p": 70},
]

# -------------------------
# Drive paths and artifact behavior
# -------------------------
TM_DIR = "/content/drive/MyDrive/TM"  # Main Google Drive folder containing input files and run outputs
ARTIFACT_BASE_DIR = f"{TM_DIR}/llm-clustering-runs"  # Folder where full run directories are saved

CLUSTER_ZIP_PATH = f"{TM_DIR}/cluster_tai.zip"  # Zip file containing clustering benchmark instances
KMEANS_RES_PATH = f"{TM_DIR}/kmeans.res"       # Reference file used for SSE and p-median baselines/references
RADIUS_REFERENCE_PATH = f"{TM_DIR}/generator_radius_reference_last_p.zip"  # Run C reference zip; auto-built from cluster_tai.zip using the p last points as generator/reference centers

# -------------------------
# Notebook behavior
# -------------------------
AUTO_DOWNLOAD_ARTIFACT_ZIP = True
COPY_ZIP_TO_DRIVE = True


## 2. Mount Google Drive

The benchmark files are read from `TM_DIR`, and run artifacts are saved under `ARTIFACT_BASE_DIR`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone or refresh the GitHub backend repo

The repository is public, so the notebook clones it directly from GitHub.


In [ ]:
repo_dir = f"/content/{REPO}"
repo_url = f"https://github.com/{ORG}/{REPO}.git"

# Move out of the repo before deleting/recloning it.
%cd /content
!rm -rf "{repo_dir}"
!git clone --branch "{BRANCH}" "{repo_url}" "{repo_dir}"

%cd "{repo_dir}"
!ls


## 4. Install backend package and optionally run tests

This installs the local package in editable mode without upgrading Colab runtime packages.

Important: we intentionally use `--no-deps` here to avoid Colab restart prompts caused by upgrading packages such as `ipython`, `ipykernel`, `tornado`, or `prompt-toolkit`. The notebook only installs truly missing minimal dependencies.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

# Make imports work immediately even before/without editable install refresh.
REPO_DIR = Path.cwd()
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Install the repo itself without dependencies. This avoids Colab runtime restart prompts
# caused by pip trying to upgrade IPython/kernel-related packages.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    check=True,
)

# Install only truly missing minimal runtime dependencies. Do not upgrade packages already
# available in Colab.
required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "requests": "requests",
    "yaml": "PyYAML",
    "psutil": "psutil",
}
missing_packages = [pkg for module, pkg in required_modules.items() if importlib.util.find_spec(module) is None]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
else:
    print("All minimal runtime dependencies already available.")

# Optional tests. Disabled by default because this is a launcher notebook, not CI.
if globals().get("RUN_TESTS_AFTER_INSTALL", False):
    if importlib.util.find_spec("pytest") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Skipping pytest because RUN_TESTS_AFTER_INSTALL = False")

# Sanity check that the backend package is visible.
import llm_clustering
print("llm_clustering import OK from:", llm_clustering.__file__)


## 5. Build runtime config from the control panel

This cell intentionally stays short. The conversion from the top control-panel variables to a temporary YAML file lives in `src/llm_clustering/notebook_runtime.py`.

In [ ]:
import sys
import importlib
from pathlib import Path

REPO_DIR = Path("/content/llm-clustering-heuristics")
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import llm_clustering.notebook_runtime as notebook_runtime
importlib.reload(notebook_runtime)

RUNTIME_CONFIG_PATH, EFFECTIVE_CONFIG = notebook_runtime.build_runtime_config_from_notebook_globals(globals())

print("Runtime config:", RUNTIME_CONFIG_PATH)
print("Module file used:", notebook_runtime.__file__)

for k in [
    "objective_mode",
    "center_constraint",
    "historical_family_avoidance",
    "sampling_mode",
    "sampling_max_xp",
    "sampling_mode_label",
    "candidate_timeout_s",
]:
    print(k, "=", EFFECTIVE_CONFIG.get(k))


## 6. Check required Drive files

Run A and Run B require `cluster_tai.zip` and `kmeans.res`.

Run C additionally requires the radius reference zip.


In [ ]:
from pathlib import Path
import subprocess
import sys

RUN_NAMES = {
    "A": "Run A — SSE / k-means objective / free centers",
    "B": "Run B — p-median objective / centers constrained to input points",
    "C": "Run C — radius-volume objective / centers constrained to input points",
}

required = [
    Path(CLUSTER_ZIP_PATH),
]

# kmeans.res is required for Run A/SSE and Run B/p-median references.
if RUN in {"A", "B"}:
    required.append(Path(KMEANS_RES_PATH))

# Run C uses the generator/reference centers: the p last points in each cluster_tai instance.
# If the reference zip does not exist yet, build it automatically from cluster_tai.zip.
if RUN == "C":
    ref_path = Path(RADIUS_REFERENCE_PATH)
    if not ref_path.exists():
        print("Run C reference missing; building generator-last-p reference automatically.")
        cmd = [
            sys.executable,
            "scripts/build_generator_radius_reference.py",
            "--cluster-zip", CLUSTER_ZIP_PATH,
            "--output-zip", str(ref_path),
        ]
        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
    required.append(ref_path)

print("Checking required files for", RUN_NAMES[RUN])
missing = []
for p in required:
    ok = p.exists()
    print(("OK      " if ok else "MISSING "), p)
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing required Drive files:\n" + "\n".join(missing))

print("All required files found.")


## 7. Load Groq API keys

This cell first tries Colab Secrets via `google.colab.userdata`, then falls back to manual input.

Expected secret names are `GROQ_API_KEY_1`, `GROQ_API_KEY_2`, etc.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

key_names = [f"GROQ_API_KEY_{i}" for i in range(1, int(GROQ_MAX_KEYS) + 1)]
loaded = []

for key_name in key_names:
    if os.environ.get(key_name):
        loaded.append(key_name)
        continue

    val = None
    if userdata is not None:
        try:
            val = userdata.get(key_name)
        except Exception:
            val = None

    if val:
        os.environ[key_name] = val
        loaded.append(key_name)

print("Loaded Groq keys from environment/Colab Secrets:", loaded)

if not os.environ.get("GROQ_API_KEY_1"):
    os.environ["GROQ_API_KEY_1"] = getpass("Paste GROQ_API_KEY_1 manually: ")

print("GROQ_API_KEY_1 found:", bool(os.environ.get("GROQ_API_KEY_1")))


## 8. Run the pipeline with live logs

This streams the backend logs while the pipeline runs, including the search/probe gap feedback printed after each LLM attempt.

In [ ]:
import subprocess
import sys

RUN_NAMES = {
    "A": "Run A — SSE / k-means objective / free centers",
    "B": "Run B — p-median objective / centers constrained to input points",
    "C": "Run C — radius-volume objective / centers constrained to input points",
}

print("Launching pipeline for", RUN_NAMES[RUN])
print("Using runtime config:", RUNTIME_CONFIG_PATH)
print("Live pipeline logs will appear below.")
print("-" * 100)

cmd = [sys.executable, "-u", "scripts/run_unified_pipeline.py", "--config", RUNTIME_CONFIG_PATH]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("-" * 100)
print("Pipeline return code:", return_code)

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)


## 9. Locate latest run folder and summarize artifacts

This finds the latest run folder matching the selected run/objective and lists the main generated artifacts.


In [ ]:
from pathlib import Path

OBJECTIVE_PREFIX = {
    "A": "llm_clustering_unified_ABC_sse_",
    "B": "llm_clustering_unified_ABC_pmedian_",
    "C": "llm_clustering_unified_ABC_radius_",
}

runs_dir = Path(ARTIFACT_BASE_DIR)
print("Runs directory:", runs_dir)

if not runs_dir.exists():
    raise FileNotFoundError(f"Runs directory does not exist: {runs_dir}")

prefix = OBJECTIVE_PREFIX.get(RUN, "llm_clustering_unified_ABC_")
matching_dirs = [p for p in runs_dir.iterdir() if p.is_dir() and p.name.startswith(prefix)]
all_dirs = [p for p in runs_dir.iterdir() if p.is_dir()]

if matching_dirs:
    latest_run_dir = max(matching_dirs, key=lambda p: p.stat().st_mtime)
elif all_dirs:
    latest_run_dir = max(all_dirs, key=lambda p: p.stat().st_mtime)
else:
    raise FileNotFoundError(f"No run folders found in {runs_dir}")

LATEST_RUN_DIR = latest_run_dir
print("Latest run dir:", LATEST_RUN_DIR)

interesting_suffixes = {".csv", ".jsonl", ".json", ".txt", ".py"}
for p in sorted(LATEST_RUN_DIR.rglob("*")):
    if p.is_file() and p.suffix.lower() in interesting_suffixes:
        print(p.relative_to(LATEST_RUN_DIR))


## 10. Quick result tables

This displays the most important summary CSVs if they exist.


In [ ]:
import pandas as pd
from IPython.display import display

summary_files = [
    "llm_attempts.csv",
    "llm_family_summary.csv",
    "llm_search_instance_rows.csv",
    "llm_probe_instance_rows.csv",
    "llm_best_attempts_top20.csv",
]

for name in summary_files:
    path = LATEST_RUN_DIR / name
    if path.exists():
        print("\n===", name, "===")
        df = pd.read_csv(path)
        display(df.head(20))
    else:
        print("Missing:", name)


## 11. Zip and auto-download latest artifact folder

If `AUTO_DOWNLOAD_ARTIFACT_ZIP = True`, this creates a zip of the latest run folder and triggers a browser download to your local PC.

If `COPY_ZIP_TO_DRIVE = True`, the zip is also copied back into `ARTIFACT_BASE_DIR`.


In [ ]:
from pathlib import Path
import shutil

from google.colab import files

if AUTO_DOWNLOAD_ARTIFACT_ZIP:
    zip_base = Path("/content") / LATEST_RUN_DIR.name
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=LATEST_RUN_DIR.parent, base_dir=LATEST_RUN_DIR.name))

    print("Created zip:", zip_path)
    print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))

    if COPY_ZIP_TO_DRIVE:
        drive_zip = Path(ARTIFACT_BASE_DIR) / zip_path.name
        shutil.copy2(zip_path, drive_zip)
        print("Copied zip to Drive:", drive_zip)

    print("Starting browser download...")
    files.download(str(zip_path))
else:
    print("AUTO_DOWNLOAD_ARTIFACT_ZIP = False, skipping local download.")
    print("Latest run folder:", LATEST_RUN_DIR)
